In [1]:
version = "REPLACE_PACKAGE_VERSION"

# Assignment 3: Mining Vectors and Matrices (Part IV)

## Patterns in Matrix Data
In the last part of the assignment, let's explore the patterns in a matrix as a whole instead of in individual vectors. As discussed in class, one type of such patterns are *eigenvectors*, which can be obtained through *Singular Value Decomposition* (SVD):

$$X=U\Sigma V^T.$$

$X$ is an $n \times p$ matrix, $U\cdot U^T = I$, $V\cdot V^T = I$, and $\Sigma$ is an $n\times p$ diagonal matrix with non-zero singular values.

Let's first walk through the example in the lecture.

In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD

from mads.lib.path import assets

np.set_printoptions(precision=6)
np.set_printoptions(suppress=True)

In [3]:
doc_word_df = pd.DataFrame([[1, 1, 1, 0, 0], 
                            [2, 2, 2, 0, 0],
                            [1, 1, 1, 0, 0],
                            [5, 5, 5, 0, 0],
                            [0, 0, 0, 2, 2],
                            [0, 0, 0, 3, 3],
                            [0, 0, 0, 1, 1]])
doc_word_df.columns = ['data', 'information', 'retrieval', 'brain', 'lung']
doc_word_df.index = ['Data Science 1', 'Data Science 2', 'Data Science 3, ', 'Data Science 4', 
                     'Medical Report 1', 'Medical Report 1', 'Medical Report 1']
doc_word_df

,data,information,retrieval,brain,lung
Data Science 1,1,1,1,0,0
Data Science 2,2,2,2,0,0
"Data Science 3,",1,1,1,0,0
Data Science 4,5,5,5,0,0
Medical Report 1,0,0,0,2,2
Medical Report 1,0,0,0,3,3
Medical Report 1,0,0,0,1,1


Several Python packages, such as NumPy, SciPy, and scikit-learn, all provide APIs for SVD. In this assignment, we will use `TruncatedSVD` API from scikit-learn, as it allows us to compute only the largest $k$ singular values in $\Sigma$ and the corresponding $k$ columns (rows) of $U$ ($V^T$), which is more efficient and practical in real-world applications. You can learn more about the API from its [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html).

For this example, we explicitly specify $k=2$ (so we are only interested in the first 2 components), and we assign a constant to `random_state` to ensure that the result is reproducible. 

In [4]:
doc_word_svd = TruncatedSVD(n_components=2, random_state=0)
doc_word_svd.fit(doc_word_df)

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",2
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",0
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


Now the object `doc_word_svd` stores all the necessary results of the decomposed matrix. You can view the singular values (the diagonal values in $\Sigma$) in the `singular_values_` attribute and the row vectors of $V^T$ in the `components_` attribute.

In [5]:
sigma_diag = doc_word_svd.singular_values_
v_t = doc_word_svd.components_
print(sigma_diag)
print(v_t)

[9.64365076 5.29150262]
[[ 5.77350269e-01  5.77350269e-01  5.77350269e-01  0.00000000e+00
   0.00000000e+00]
 [ 1.79118808e-20 -8.95594041e-21 -8.95594041e-21  7.07106781e-01
   7.07106781e-01]]


You may wonder where the $U$ matrix is. Although $U$ is not explicitly stored in the `TruncatedSVD` object. it can be recovered with $X$, $\Sigma$, and $V$, based on the following formula:

$$ X^* = U\Sigma = XV.$$

Thus, $U$ can be written as

$$ U = X^* \Sigma^{-1} = XV\Sigma^{-1}. $$

We can verify this with the following code:

In [6]:
x = doc_word_df.values
v = v_t.T
sigma_inverse = np.linalg.inv(np.diag(sigma_diag))

u = x.dot(v).dot(sigma_inverse)
print(u)

[[1.79605302e-01 2.55913979e-36]
 [3.59210604e-01 5.11827958e-36]
 [1.79605302e-01 2.55913979e-36]
 [8.98026510e-01 1.27956990e-35]
 [0.00000000e+00 5.34522484e-01]
 [0.00000000e+00 8.01783726e-01]
 [0.00000000e+00 2.67261242e-01]]


Now let's try to interpret $U$, $\Sigma$, and $V^T$ by thinking about the following questions (not graded):

* Which codes the Data Science / Medical concept of each document?
* Which codes the strength of each concept?
* Which codes the word representation of each concept?

One important application of SVD is to transform the original matrix $X$ in to a new matrix $X^*$. This transformation projects the $p$-dimensional row vectors in the original matrix into $k$-dimensional vectors in the new vector space. $X^*$ encodes most of the information in the original $X$ matrix with fewer dimensions.

Mathamatically, the transformation is made by post-multiplying $X$ with the $V$ matrix ($ X^* = XV$). The `TruncatedSVD` API has implemented such transformation as a `transform` function. You may verify this with the following code:

In [7]:
print(doc_word_svd.transform(doc_word_df))

[[1.73205081e+00 1.35416949e-35]
 [3.46410162e+00 2.70833898e-35]
 [1.73205081e+00 1.35416949e-35]
 [8.66025404e+00 6.77084746e-35]
 [0.00000000e+00 2.82842712e+00]
 [0.00000000e+00 4.24264069e+00]
 [0.00000000e+00 1.41421356e+00]]


In [8]:
print(doc_word_df.values.dot(v))

[[1.73205081e+00 1.35416949e-35]
 [3.46410162e+00 2.70833898e-35]
 [1.73205081e+00 1.35416949e-35]
 [8.66025404e+00 6.77084746e-35]
 [0.00000000e+00 2.82842712e+00]
 [0.00000000e+00 4.24264069e+00]
 [0.00000000e+00 1.41421356e+00]]


Now that you are familiar with the SVD operations. Let's apply it to the Montréal restaurant dataset. Run the following code block to load the dataset prepared in Part I of this assignment.

In [9]:
file_business = assets.find("montreal_business.csv")
file_user = assets.find("montreal_user.csv")
business_df = pd.read_csv(file_business)
business_df.set_index('business_id', inplace=True)

review_df = pd.read_csv(file_user)
rating_df = review_df.pivot_table(index=['business_id'], columns=['user_id'], values='stars')
rating_df.fillna(0, inplace=True)

missing_business_id = set(business_df.index) - set(rating_df.index)
business_df.drop(missing_business_id, inplace=True)

### Exercise 5. (15 pts.)
Complete the following `transformed_rating` function, which transforms the original $n\times p$ rating matrix into a new $n \times k$ matrix using the `TruncatedSVD` API. The function should return the $n \times k$ matrix.

**Note:**
1. Please set the `random_state` to 0 so that the results do not change over different runs.
2. $k$ can take values between 1 and $p$, not necessarily the 2 used in the document-word matrix example.
3. You may either use the `transformation` function in the `TruncatedSVD` object or use the dot product. Both are OK.

In [10]:
def svd_transformed_rating(original_matrix, k, random_state=0):

    # fit truncated SVD on the original matrix
    svd = TruncatedSVD(n_components = k, random_state = random_state)

    # project the original matrix into the lower-dimensional space
    transformed_matrix = svd.fit_transform(original_matrix)
    
    return transformed_matrix

With the `svd_transformed_rating` function, we can calculate the transformed matrix through the following command.

In [11]:
rating_matrix_d2 = svd_transformed_rating(rating_df, 2)
# You can uncomment the following line to view the result
rating_matrix_d2

array([[26.83083536,  6.96003359],
       [ 0.24013478, -0.0319724 ],
       [ 2.97134101,  2.14146709],
       ...,
       [ 0.29149761, -0.05069042],
       [ 3.21725035,  2.32169523],
       [ 1.50347684,  1.01851118]], shape=(2770, 2))

In [12]:
# This code block tests if the `svd_transformed_rating` function is implemented correctly.
# We hide some tests, so passing all the displayed assertions does not guarantee full points.

rating_matrix_d2 = svd_transformed_rating(rating_df, 2)
# d2_groundtruth is calculated via the following command:
# d2_groundtruth = svd_transformed_rating(rating_df, 2)[:5]
d2_groundtruth = np.array([[26.830835356   , 6.960033590895],
                           [ 0.240134782174, -0.031972398386],
                           [ 2.971341010574, 2.141467087847],
                           [ 0.997482022276, 0.599895804555],
                           [ 0.470445852682, 0.482343299048]])
assert np.allclose(rating_matrix_d2[:5], d2_groundtruth)

rating_matrix_d5 = svd_transformed_rating(rating_df, 5)
d5_groundtruth = np.array([[26.828890919511, 6.950258673617, -1.515366172479, 3.295739836372,  4.157186247066],
                           [ 0.24012936551 , -0.031884190966, 0.018132185754, 0.166903622477, -0.028071803934],
                           [ 2.9713046863  , 2.141818808721, 1.232252381363, 0.691324426529,  0.11961284882 ],
                           [ 0.997484724453, 0.59970688578 , 0.107441873278, -0.047329773205,  0.160751420569],
                           [ 0.470427947552, 0.482204394535, -0.062480138124, 0.238280610506,  0.055425249737]])
assert np.allclose(rating_matrix_d5[:5], d5_groundtruth)

rating_matrix_transformed = svd_transformed_rating(rating_df, 100)
assert rating_matrix_transformed.shape == (2770,100), "The transformed matrix is of the wrong dimension."
for i in range(100):
    for j in range(i):
        assert abs(rating_matrix_transformed.T[i].dot(rating_matrix_transformed.T[j])) < 1e-8, \
            "The column vectors in X* should be orthogonal."


A quick note on the above test cell: Ideally, the first $k'$ columns should stay the same for any $k$ as long as $k > k'$. However, you may notice small differences in the test cell (e.g. 26.830835356 vs. 26.828890919511). This is because the `TruncatedSVD` API uses a randomized SVD solver to speed up the calculation, so the output may lose minor precision.

As you can see, we even increased $k$ to 100, which is much larger than $k=2$ used in the document-word example. However, this is still much smaller than the 11,937 dimensions in the original rating matrix, and we shall soon see that these 100 dimensions have preserved most of the information in the original matrix.

### Exercise 6. (10 pts).
This exercise is to help you examine the power of SVD. That is, we want to see how the 100-dimension vectors preserve much information from the 11,937-dimensional vectors. Indeed, we shall see that the similarity between vectors is preserved after the SVD transformation. To show that, we are going to combine several techniques we have learned so far, including vector similarity, patterns in matrix data, and itemset similarity.

Please complete the following task:
1. Complete the `find_max_dot_prod_restaurants` and the `find_max_dot_prod_restaurants_transformed` function. Each function returns the `business_id` of the top-$n$ restaurants that have the largest dot product with Modavie. The `find_max_dot_prod_restaurants` function calculates the dot product based on the raw vectors (11,937 dimensions). The  `find_max_dot_prod_restaurants_transformed` function calculates with the transformed vectors (100 dimensions). You may copy and paste `find_max_dot_prod_restaurants` from Part III of this assignment.
2. Complete the `jaccard_similarity_before_after_svd_transform` function, which calculates the Jaccard similarity between the top-$n$ most similar restaurants (of course, this is an itemset!) before and after the SVD transformation.

We have provided all the test cases to help you verify your results.

In [13]:
# This cell helps you prepare a new rating dataframe using the transformed vectors.
rating_matrix_transformed = svd_transformed_rating(rating_df, 100)
rating_df_transformed = pd.DataFrame(rating_matrix_transformed, index=rating_df.index)
rating_df_transformed.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
business_id,,,,,,,,,,,,,,,,,,,,,
-1xuC540Nycht_iWFeJ-dw,26.829882,6.949084,-1.637419,3.495390,4.253959,1.578638,4.723778,-39.660973,18.228902,-23.421889,...,0.767882,-1.533552,-1.182358,0.950001,0.017549,-0.262306,1.707314,1.132702,-0.558816,-0.669832
-7bRnaHp7OHz8KW-THqP4w,0.240134,-0.031948,0.017022,0.169980,-0.026894,0.095219,0.032490,-0.103308,0.170128,0.030196,...,0.225307,-0.001314,0.318106,0.128878,0.100352,-0.165589,-0.204660,-0.075091,0.270025,0.273088
-92cC6-X87HQ1DE1UHOx3w,2.971325,2.141960,1.232110,0.698304,0.111484,0.469199,-0.181528,1.522476,-0.333825,-0.420363,...,1.406269,-1.265186,1.808670,0.257573,0.417470,0.545357,-1.153764,-0.511001,-0.184996,1.439649
-AgfhwHOYrsPKt-_xV_Ipg,0.997471,0.599784,0.108910,-0.053457,0.161304,-0.202928,0.815409,-0.270971,-0.649032,-0.253430,...,0.384626,1.344930,-0.325934,0.000308,0.328897,-0.074603,-0.192419,-0.501441,-0.059467,0.565910
-FDkvLmwaBrtVgYFqEWeWA,0.470432,0.481998,-0.060954,0.237180,0.047426,-0.081098,0.075756,0.257721,-0.141577,-0.436627,...,-0.100854,0.031623,0.198573,-0.596779,-0.659795,-0.022381,0.060685,-0.787100,-0.165884,-0.032865


In [21]:
def find_max_dot_prod_restaurants(top_n):
    modavie_id = business_df[business_df.name.str.contains("Modavie")].index[0]
    modavie_vector = rating_df.loc[modavie_id]

    # compute dot product between every restaurant and Modavie
    dot_prod = rating_df.dot(modavie_vector)

    # remove Modavie itself
    dot_prod = dot_prod.drop(modavie_id)

    # sort descending and keep the top n
    top_restaurants = dot_prod.sort_values(ascending = False).head(top_n)

    # return a dataframe that includes the restaurant names
    result = business_df.loc[top_restaurants.index, ['name']].copy()
    result['dot_prod'] = top_restaurants.values

    # keep the dataframe sorted for the autograder
    result = result.sort_values('dot_prod', ascending = False)

    return result 

def find_max_dot_prod_restaurants_transformed(top_n):
    modavie_id = business_df[business_df.name.str.contains("Modavie")].index[0]
    modavie_vector_transformed = rating_df_transformed.loc[modavie_id]

    # compute dot product in the transformed space
    dot_prod = rating_df_transformed.dot(modavie_vector_transformed)

    # remove Modavie itself
    dot_prod = dot_prod.drop(modavie_id)

    # sort descendiing and keep the top n
    top_restaurants = dot_prod.sort_values(ascending= False).head(top_n)

    # return a dataframe that includes the restaurants names
    result = business_df.loc[top_restaurants.index, ['name']].copy()
    result['dot_prod'] = top_restaurants.values

    # keep the dataframe sorted  for the autograder
    result = result.sort_values('dot_prod', ascending = False)

    return result 

In [22]:
# This code block help you verify if `find_max_dot_prod_restaurants_transformed` is implemented correctly.
answer = find_max_dot_prod_restaurants_transformed(10)
assert answer.iloc[0]['name'] == "Schwartz's"
assert answer.iloc[1]['name'] == "La Banquise"
assert answer.iloc[2]['name'] == "Olive & Gourmando"
assert answer.iloc[3]['name'] == "Maison Christian Faure"
assert answer.iloc[4]['name'] == "Reuben's Deli & Steakhouse"

assert answer.iloc[5]['name'] == "Kazu"
assert answer.iloc[6]['name'] == "Belon"
assert answer.iloc[7]['name'] == "Au Pied de Cochon"
assert answer.iloc[8]['name'] == "L'Avenue"
assert answer.iloc[9]['name'] == "Poutineville"

In [23]:
def jaccard_similarity_before_after_svd_transform(top_n):
    max_sim_restaurants = find_max_dot_prod_restaurants(top_n)
    max_sim_restaurants_transformed = find_max_dot_prod_restaurants_transformed(top_n)
    
    # construct the set of the names of the most similar restaurants
    business_id_before = set(max_sim_restaurants.name)
    business_id_after = set(max_sim_restaurants_transformed.name)
    
    # compute the Jaccard similarity between the two set and return the Jaccard similarity
    jaccard_similarity = len(business_id_before.intersection(business_id_after)) / len(business_id_before.union(business_id_after))
    
    return jaccard_similarity

In [24]:
# This code block checks `find_max_dot_prod_restaurants`, `find_max_dot_prod_restaurants_transformed`, 
# and 'jaccard_similarity_before_after_svd_transform'. You should get the correct answer if all three 
# functions are implemented correctly.
# We hide some tests, so passing all the displayed assertions does not guarantee full points.

assert abs(jaccard_similarity_before_after_svd_transform(5) - 1) < 1e-8
assert (abs(jaccard_similarity_before_after_svd_transform(10) - 9 / 11) < 1e-8) or (abs(jaccard_similarity_before_after_svd_transform(10) - 2 / 3) < 1e-8)
assert abs(jaccard_similarity_before_after_svd_transform(2770) - 1) < 1e-8


As you can see, you can perform many tasks on the transformed vectors just as what you can do on the raw vectors. Since the transformed vectors have far fewer dimensions, the efficiency is much higher with the transformed vectors. Beyond efficiency, dimension reduction has many other benefits for dealing with matrix data.  You will learn these in the downstream machine learning classes, but let's stop here for now.  Matrix analysis has lots of applications in recommender systems.  We will revisit some of these techniques when that topic comes up.  

This concludes this assignment.